In [11]:
import sqlite3
import pandas as pd

# Load CSVs
users_df = pd.read_csv("users.csv")
events_df = pd.read_csv("events.csv")

# Create SQLite DB
conn = sqlite3.connect("saas_data.db")

# Push dataframes into SQL tables
users_df.to_sql("users", conn, if_exists="replace", index=False)
events_df.to_sql("events", conn, if_exists="replace", index=False)

print("Database created successfully!")
print(f"Users table: {len(users_df)} rows")
print(f"Events table: {len(events_df)} rows")

Database created successfully!
Users table: 500 rows
Events table: 13710 rows


In [12]:
#Query 1 — Monthly Active Users (MAU)

query1 = """
SELECT 
    strftime('%Y-%m', event_date) AS month,
    COUNT(DISTINCT user_id) AS active_users
FROM events
GROUP BY month
ORDER BY month;
"""
pd.read_sql(query1, conn)

,month,active_users
0,2025-04,5
1,2025-05,40
2,2025-06,72
3,2025-07,111
4,2025-08,147
5,2025-09,179
6,2025-10,217
7,2025-11,254
8,2025-12,288
9,2026-01,324


In [13]:
#Query 2 — Most & least used features

query2 = """
SELECT 
    feature,
    COUNT(*) AS total_usage,
    COUNT(DISTINCT user_id) AS unique_users
FROM events
GROUP BY feature
ORDER BY total_usage DESC;
"""
pd.read_sql(query2, conn)

,feature,total_usage,unique_users
0,dashboard,4924,449
1,reports,3340,415
2,api_access,1360,343
3,team_collab,1355,347
4,integrations,1112,312
5,export,949,294
6,notifications,670,251


In [14]:
#Query 3 — Feature usage by plan (free vs starter vs pro)

query3 = """
SELECT 
    plan,
    feature,
    COUNT(*) AS usage_count
FROM events
GROUP BY plan, feature
ORDER BY plan, usage_count DESC;
"""
pd.read_sql(query3, conn)

,plan,feature,usage_count
0,free,dashboard,808
1,free,reports,509
2,free,team_collab,229
3,free,api_access,225
4,free,integrations,179
5,free,export,137
6,free,notifications,97
7,pro,dashboard,2545
8,pro,reports,1784
9,pro,api_access,718


In [15]:
#Query 4 — At-risk users (no activity in 30+ days)

query4 = """
SELECT 
    u.user_id,
    u.plan,
    u.signup_date,
    MAX(e.event_date) AS last_active,
    julianday('now') - julianday(MAX(e.event_date)) AS days_inactive
FROM users u
JOIN events e ON u.user_id = e.user_id
GROUP BY u.user_id
HAVING days_inactive > 30
ORDER BY days_inactive DESC;
"""
pd.read_sql(query4, conn)

,user_id,plan,signup_date,last_active,days_inactive
0,U1076,pro,2025-05-30,2025-09-10,228.799513
1,U1047,free,2025-04-27,2025-11-05,172.799513
2,U1208,free,2025-07-27,2025-11-09,168.799513
3,U1237,free,2025-11-09,2025-11-24,153.799513
4,U1257,free,2025-07-31,2025-11-25,152.799513
...,...,...,...,...,...
81,U1333,free,2025-11-16,2026-03-27,30.799513
82,U1296,free,2025-09-06,2026-03-27,30.799513
83,U1288,free,2026-03-01,2026-03-27,30.799513
84,U1205,free,2026-02-14,2026-03-27,30.799513


In [16]:
#Query 5 — Average time to first action after signup

query5 = """
SELECT 
    u.plan,
    ROUND(AVG(julianday(e.first_action) - julianday(u.signup_date)), 1) AS avg_days_to_first_action
FROM users u
JOIN (
    SELECT user_id, MIN(event_date) AS first_action
    FROM events
    GROUP BY user_id
) e ON u.user_id = e.user_id
GROUP BY u.plan;
"""
pd.read_sql(query5, conn)

,plan,avg_days_to_first_action
0,free,30.1
1,pro,7.9
2,starter,9.8
